# L22: API 服务开发

> 使用 FastAPI 构建 Agent 后端服务

In [ ]:
# === 环境设置 ===
import sys
import os
import json
import asyncio
from pathlib import Path
from typing import List, Dict, Any, Optional

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

# 查看项目 API 结构
import backend.api
print("✓ API 模块导入成功")

## 22.1 FastAPI 简介

### 为什么选择 FastAPI？

| 特性 | 说明 |
|------|------|
| **高性能** | 与 NodeJS 和 Go 相当 |
| **异步支持** | 原生 async/await |
| **自动文档** | Swagger UI 开箱即用 |
| **类型验证** | Pydantic 自动校验 |
| **易学易用** | 直观的装饰器语法 |
| **Agent 友好** | 与 LLM 调用完美配合 |

### 项目中的 API 结构

In [ ]:
# 查看项目 API 结构
import os

api_dir = Path(project_path) / "backend" / "api"
print("=== API 目录结构 ===\n")
 
for root, dirs, files in os.walk(api_dir):
    level = root.replace(str(api_dir), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        if not file.startswith('__'):
            print(f'{subindent}{file}')

## 22.2 基础 API 端点

In [ ]:
# FastAPI 基础示例
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional

# 创建 FastAPI 应用
app = FastAPI(
    title="Agent Learning API",
    description="Agent 学习项目的后端 API",
    version="1.0.0"
)

# 请求/响应模型
class ChatRequest(BaseModel):
    message: str
    session_id: Optional[str] = None
    temperature: Optional[float] = 0.7
    
class ChatResponse(BaseModel):
    response: str
    session_id: str
    tokens_used: int

# 基础端点
@app.get("/")
async def root():
    """根路径"""
    return {
        "message": "Agent Learning API",
        "version": "1.0.0",
        "docs": "/docs"
    }

@app.get("/health")
async def health_check():
    """健康检查"""
    return {"status": "healthy"}

# 打印路由
print("=== FastAPI 路由 ===\n")
for route in app.routes:
    if hasattr(route, 'path') and hasattr(route, 'methods'):
        methods = ', '.join(sorted(route.methods))
        print(f"{methods:10} {route.path}")

## 22.3 Agent API 端点

In [ ]:
# Agent API 设计

class AgentConfig(BaseModel):
    """Agent 配置"""
    provider: str = "deepseek"
    model: Optional[str] = None
    temperature: float = 0.7
    max_tokens: int = 2000
    system_prompt: Optional[str] = None

class AgentRequest(BaseModel):
    """Agent 请求"""
    query: str
    config: Optional[AgentConfig] = None
    session_id: Optional[str] = None

# 端点定义（示例）
api_endpoints = {
    "POST /api/chat": "单轮对话",
    "POST /api/chat/stream": "流式对话",
    "POST /api/agent/run": "运行 ReAct Agent",
    "GET /api/sessions": "获取会话列表",
    "GET /api/sessions/{id}": "获取会话详情",
    "DELETE /api/sessions/{id}": "删除会话",
    "GET /api/tools": "获取可用工具",
    "POST /api/tools/execute": "执行工具",
}

print("=== Agent API 端点设计 ===\n")
for endpoint, description in api_endpoints.items():
    print(f"{endpoint:35} - {description}")

## 22.4 查看项目 API 实现

In [ ]:
# 查看主 API 文件
api_main_path = Path(project_path) / "backend" / "api" / "main.py"

if api_main_path.exists():
    print("=== main.py 源码 ===\n")
    with open(api_main_path, encoding="utf-8") as f:
        content = f.read()
        print(content[:1500] + "\n... (源码已截断)")
else:
    print("main.py 文件不存在")

## 22.5 API 数据模型

In [ ]:
# 查看项目的 API schemas
schemas_path = Path(project_path) / "backend" / "api" / "schemas.py"

if schemas_path.exists():
    print("=== API Schemas ===\n")
    with open(schemas_path, encoding="utf-8") as f:
        content = f.read()
        print(content[:1000] + "\n... (源码已截断)")
else:
    print("schemas.py 文件不存在")

## 22.6 运行 API 服务

In [ ]:
# API 服务启动命令
print("=== 启动 API 服务 ===\n")

print("开发模式（自动重载）:")
print("  uvicorn backend.api.main:app --reload")

print("\n生产模式:")
print("  uvicorn backend.api.main:app --host 0.0.0.0 --port 8000")

print("\n使用 uv 运行:")
print("  uv run uvicorn backend.api.main:app --reload")

print("\n访问地址:")
print("  - API: http://localhost:8000")
print("  - 文档: http://localhost:8000/docs")
print("  - Redoc: http://localhost:8000/redoc")

## 22.7 错误处理和中间件

In [ ]:
from fastapi import Request, status
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError

# 自定义异常
class AgentException(Exception):
    """Agent 基础异常"""
    def __init__(self, message: str, code: str = "AGENT_ERROR"):
        self.message = message
        self.code = code

# 异常处理器
async def agent_exception_handler(request: Request, exc: AgentException):
    """处理 Agent 异常"""
    return JSONResponse(
        status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
        content={
            "error": True,
            "code": exc.code,
            "message": exc.message
        }
    )

async def validation_exception_handler(request: Request, exc: RequestValidationError):
    """处理验证异常"""
    return JSONResponse(
        status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
        content={
            "error": True,
            "code": "VALIDATION_ERROR",
            "message": "请求参数验证失败",
            "details": exc.errors()
        }
    )

# CORS 中间件配置
cors_config = {
    "allow_origins": ["*"],  # 生产环境应限制具体域名
    "allow_credentials": True,
    "allow_methods": ["*"],
    "allow_headers": ["*"],
}

print("=== 错误处理和中间件配置 ===\n")
print("异常处理器:")
print("  - AgentException → 500 错误")
print("  - RequestValidationError → 422 错误")
print("\nCORS 配置:")
for k, v in cors_config.items():
    print(f"  {k}: {v}")

## 22.8 常见面试问题

**Q: FastAPI 和 Flask 有什么区别？**
- A:
  | 特性 | FastAPI | Flask |
  |------|---------|-------|
  | 性能 | 更高（异步） | 较低（同步） |
  | 类型检查 | 自动 Pydantic | 手动 |
  | 文档 | 自动生成 | 需要手动配置 |
  | 异步 | 原生支持 | 需要扩展 |
  | 学习曲线 | 稍陡 | 平缓 |

**Q: 如何保证 API 的安全性？**
- A:
  1. **身份验证**：JWT、OAuth2
  2. **输入验证**：Pydantic 模型
  3. **CORS 限制**：只允许可信域名
  4. **速率限制**：防止滥用
  5. **HTTPS**：加密传输
  6. **敏感数据保护**：不暴露 API Key

**Q: 如何处理 LLM 调用的异步特性？**
- A:
  1. 使用 `async def` 定义端点
  2. 使用 `await` 调用异步 LLM 方法
  3. FastAPI 自动处理事件循环
  4. 并发请求会被正确处理
  5. 可以使用 `asyncio.gather` 并行处理

---

## 总结

本课程学习了 API 服务开发的核心知识：

| 知识点 | 说明 |
|--------|------|
| **FastAPI** | 高性能异步 Web 框架 |
| **路由设计** | RESTful API 端点设计 |
| **数据模型** | Pydantic 请求/响应模型 |
| **错误处理** | 自定义异常和处理器 |
| **中间件** | CORS、日志、认证 |
| **异步支持** | async/await 处理 LLM 调用 |

**下一步**: L23 将学习前端界面开发！